# Day 7 Exam: Integration & Interview Prep

**Coverage:** Notebooks 18-21 (long context, multimodal generation, finetuning/adapters, distributed training), with heavy review of all previous days. This is the final exam before interview.

---

## Exam Rules

- **Total time: ~2 hours** (warmups ~15 min, main problems ~90 min)
- **No documentation, no Google, no LLM assistance.** Closed-book.
- **Honor system.** If you can't remember something, leave a comment and move on.
- Write clean, typed, documented code — as if submitting for code review.
- Run all test/assertion cells to validate your work before finishing.
- If stuck on a warmup for >7 min, move on. Main problems matter more.
- **This is the final checkpoint.** Treat it like an actual interview coding session.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import math
import os

---

## Warmup Section (~15 minutes total)

Quick recall problems. 5 minutes each. Don't overthink these.

### Warmup 1: LoRA Forward Pass (5 min)

Implement `lora_forward(x, W, A, B, alpha, rank)` as a **standalone function** (no `nn.Module`).

**Formula:** `output = x @ W.T + (alpha / rank) * x @ A.T @ B.T`

**Shapes:**
- `x`: `(B, in_features)` — input
- `W`: `(out_features, in_features)` — frozen pretrained weight matrix
- `A`: `(rank, in_features)` — LoRA down-projection
- `B`: `(out_features, rank)` — LoRA up-projection
- `alpha`: float — LoRA scaling factor
- `rank`: int — LoRA rank
- Output: `(B, out_features)`

**Note:** The LoRA contribution is `(alpha/rank) * B @ A @ x`, which in matrix form with x on the left is `(alpha/rank) * x @ A.T @ B.T`.

In [ ]:
# Warmup 1: Your implementation



In [ ]:
# Warmup 1: Tests
torch.manual_seed(42)
B_size, in_f, out_f, rank = 4, 64, 128, 8
alpha = 16.0

x = torch.randn(B_size, in_f)
W = torch.randn(out_f, in_f)
A = torch.randn(rank, in_f)
B_mat = torch.randn(out_f, rank)

out = lora_forward(x, W, A, B_mat, alpha, rank)
assert out.shape == (B_size, out_f), f"Expected {(B_size, out_f)}, got {out.shape}"

# B=0 should give just Wx (no LoRA contribution)
B_zero = torch.zeros(out_f, rank)
out_no_lora = lora_forward(x, W, A, B_zero, alpha, rank)
expected_base = x @ W.T
assert torch.allclose(out_no_lora, expected_base, atol=1e-5), "B=0 should give W @ x only"

# A=0 should also give just Wx
A_zero = torch.zeros(rank, in_f)
out_no_lora2 = lora_forward(x, W, A_zero, B_mat, alpha, rank)
assert torch.allclose(out_no_lora2, expected_base, atol=1e-5), "A=0 should give W @ x only"

print("Warmup 1 passed.")

### Warmup 2: DDP Info Helper (5 min)

Implement `get_ddp_info() -> dict` that reads distributed training info from environment variables.

**Reads these env vars (with defaults for non-distributed / single-GPU):**
- `WORLD_SIZE` → `world_size` (default: `1`)
- `RANK` → `rank` (default: `0`)
- `LOCAL_RANK` → `local_rank` (default: `0`)

**Returns:** `{"world_size": int, "rank": int, "local_rank": int}`

Use `os.environ.get(key, default)` and convert to `int`.

In [ ]:
# Warmup 2: Your implementation



In [ ]:
# Warmup 2: Tests
# In a non-distributed environment, defaults should apply
info = get_ddp_info()
assert isinstance(info, dict), "Should return a dict"
assert set(info.keys()) == {"world_size", "rank", "local_rank"}

# Defaults should be sensible for single-GPU
assert info["world_size"] >= 1, "world_size should be >= 1"
assert info["rank"] >= 0, "rank should be >= 0"
assert info["local_rank"] >= 0, "local_rank should be >= 0"

# All values should be ints
assert all(isinstance(v, int) for v in info.values()), "All values should be int"

# Test with explicit env vars
os.environ["WORLD_SIZE"] = "4"
os.environ["RANK"] = "2"
os.environ["LOCAL_RANK"] = "1"
info_dist = get_ddp_info()
assert info_dist["world_size"] == 4
assert info_dist["rank"] == 2
assert info_dist["local_rank"] == 1

# Clean up
del os.environ["WORLD_SIZE"]
del os.environ["RANK"]
del os.environ["LOCAL_RANK"]

print("Warmup 2 passed.")

### Warmup 3: Gradient Accumulation Step (5 min)

Implement `gradient_accumulation_step(model, batch, accumulation_steps, step_idx, optimizer) -> float`.

**Behavior:**
1. Forward pass: `output = model(batch)`, `loss = output.mean()`
2. Scale the loss: `scaled_loss = loss / accumulation_steps`
3. Backward pass: `scaled_loss.backward()`
4. If `(step_idx + 1) % accumulation_steps == 0`: call `optimizer.step()` and `optimizer.zero_grad()`
5. Return `loss.item()` (the unscaled loss value)

**Args:**
- `model`: `nn.Module`
- `batch`: input tensor
- `accumulation_steps`: int — accumulate gradients over this many steps
- `step_idx`: int — current step index (0-based)
- `optimizer`: `torch.optim.Optimizer`

In [ ]:
# Warmup 3: Your implementation



In [ ]:
# Warmup 3: Tests
torch.manual_seed(42)

model = nn.Linear(16, 4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
accumulation_steps = 4

# Track when optimizer actually steps by monitoring param changes
param_snapshots = []
for step_idx in range(8):
    batch = torch.randn(2, 16)
    params_before = model.weight.data.clone()
    loss_val = gradient_accumulation_step(model, batch, accumulation_steps, step_idx, optimizer)
    params_after = model.weight.data.clone()
    params_changed = not torch.equal(params_before, params_after)
    param_snapshots.append(params_changed)
    assert isinstance(loss_val, float), f"Should return float, got {type(loss_val)}"

# Optimizer should step only at step_idx 3 and 7 (every 4 steps)
expected_steps = [False, False, False, True, False, False, False, True]
assert param_snapshots == expected_steps, (
    f"Optimizer stepped at wrong intervals.\n"
    f"Expected: {expected_steps}\n"
    f"Got:      {param_snapshots}"
)

print("Warmup 3 passed.")

---

## Main Section (~90 minutes total)

Substantial implementation problems. 30 minutes each. Write clean, tested code.

### Main Problem 1: Full Training Pipeline from Scratch (30 min)

Implement a complete training pipeline. All components from scratch, no hints beyond the interface.

**Components to implement:**

1. **`SimpleDataset(Dataset)`** — synthetic dataset:
   - `__init__(self, n_samples: int, img_channels: int = 3, img_size: int = 32, n_classes: int = 10)`
   - Generates random image tensors `(C, H, W)` and random integer labels `[0, n_classes)`
   - `__len__` and `__getitem__` as usual

2. **`SimpleCNN(nn.Module)`** — 3-layer CNN classifier:
   - `__init__(self, img_channels: int = 3, n_classes: int = 10)`
   - Architecture: 3 conv layers (increasing channels, e.g., 3→16→32→64) each followed by ReLU and MaxPool2d(2)
   - Followed by AdaptiveAvgPool2d(1) → Flatten → Linear → output `(B, n_classes)`

3. **`train_one_epoch(model, loader, optimizer, device) -> float`**:
   - Standard training loop over one epoch
   - Uses `F.cross_entropy` loss
   - Returns average loss over the epoch

4. **`save_checkpoint(model, optimizer, epoch, path)`**:
   - Save `{"model_state_dict": ..., "optimizer_state_dict": ..., "epoch": ...}` via `torch.save`

5. **`load_checkpoint(path, model, optimizer) -> int`**:
   - Load checkpoint, restore model and optimizer states
   - Return the saved epoch number

In [ ]:
# Main Problem 1: Your implementation



In [ ]:
# Main Problem 1: Tests
import tempfile
torch.manual_seed(42)
device = torch.device("cpu")

# Test dataset
dataset = SimpleDataset(n_samples=100, img_channels=3, img_size=32, n_classes=10)
assert len(dataset) == 100
img, label = dataset[0]
assert img.shape == (3, 32, 32), f"Expected (3, 32, 32), got {img.shape}"
assert isinstance(label, int) or (isinstance(label, torch.Tensor) and label.dim() == 0)

# Test model forward
model = SimpleCNN(img_channels=3, n_classes=10).to(device)
x = torch.randn(4, 3, 32, 32).to(device)
out = model(x)
assert out.shape == (4, 10), f"Expected (4, 10), got {out.shape}"

# Test training loop — loss should decrease
loader = DataLoader(dataset, batch_size=16, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

loss_epoch1 = train_one_epoch(model, loader, optimizer, device)
loss_epoch2 = train_one_epoch(model, loader, optimizer, device)
assert loss_epoch2 < loss_epoch1, f"Loss should decrease: epoch1={loss_epoch1:.4f}, epoch2={loss_epoch2:.4f}"

# Test checkpoint round-trip
with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
    ckpt_path = f.name

save_checkpoint(model, optimizer, epoch=2, path=ckpt_path)

# Create fresh model and optimizer
model2 = SimpleCNN(img_channels=3, n_classes=10).to(device)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

# Before loading, outputs should differ
test_input = torch.randn(2, 3, 32, 32).to(device)
out_original = model(test_input)
out_fresh = model2(test_input)
# (They might coincidentally match, but very unlikely)

# Load checkpoint
loaded_epoch = load_checkpoint(ckpt_path, model2, optimizer2)
assert loaded_epoch == 2, f"Expected epoch=2, got {loaded_epoch}"

# After loading, outputs should match
out_loaded = model2(test_input)
assert torch.allclose(out_original, out_loaded, atol=1e-6), "Loaded model should match original model output"

# Clean up
os.unlink(ckpt_path)

print(f"Epoch 1 loss: {loss_epoch1:.4f}")
print(f"Epoch 2 loss: {loss_epoch2:.4f}")
print("Main Problem 1 passed.")

### Main Problem 2: LoRA Implementation (30 min)

Implement Low-Rank Adaptation (LoRA) for efficient fine-tuning.

**Components:**

1. **`LoRALinear(nn.Module)`**:
   - `__init__(self, original: nn.Linear, rank: int, alpha: float)`
     - Store the original Linear layer (freeze its parameters: `requires_grad_(False)`)
     - Create `A`: `nn.Parameter` of shape `(rank, in_features)` — initialized with Kaiming uniform
     - Create `B`: `nn.Parameter` of shape `(out_features, rank)` — initialized to **zeros** (so initial output matches original)
     - Store `alpha` and `rank`
   - `forward(self, x) -> Tensor`:
     - `return original(x) + (alpha / rank) * (x @ A.T @ B.T)`
     - If original has bias, it's already included in `original(x)`
   - `merge(self) -> nn.Linear`:
     - Return a new `nn.Linear` whose weight = `original.weight + (alpha/rank) * B @ A`
     - Copy bias from original if it exists

2. **`inject_lora(model: nn.Module, rank: int, alpha: float) -> nn.Module`**:
   - Walk the model and replace every `nn.Linear` with a `LoRALinear` wrapper
   - Return the modified model
   - Hint: use `model.named_modules()` to find Linear layers, then `setattr` on parent modules to replace them

3. **`count_trainable(model: nn.Module) -> int`**:
   - Return total number of parameters where `requires_grad == True`

In [ ]:
# Main Problem 2: Your implementation



In [ ]:
# Main Problem 2: Tests
torch.manual_seed(42)

# Test LoRALinear directly
original = nn.Linear(64, 128)
lora_layer = LoRALinear(original, rank=4, alpha=8.0)

x = torch.randn(2, 64)

# Zero-init B means output should match original
out_lora = lora_layer(x)
out_original = original(x)
assert torch.allclose(out_lora, out_original, atol=1e-5), \
    "Zero-init B should make LoRA output match original"

# Only A and B should be trainable
trainable_params = [n for n, p in lora_layer.named_parameters() if p.requires_grad]
assert len(trainable_params) == 2, f"Expected 2 trainable params (A, B), got {len(trainable_params)}: {trainable_params}"

# Test merge: after modifying B, merge should give same output as LoRA forward
with torch.no_grad():
    lora_layer.B.normal_()  # give B non-zero values
out_lora_modified = lora_layer(x)
merged = lora_layer.merge()
out_merged = merged(x)
assert torch.allclose(out_lora_modified, out_merged, atol=1e-5), \
    "Merged linear should produce same output as LoRA forward"

# Test inject_lora on a small model
class TinyTransformer(nn.Module):
    def __init__(self, dim: int = 64) -> None:
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(F.relu(self.v(x))) + x

model = TinyTransformer(dim=64)
total_before = sum(p.numel() for p in model.parameters())
trainable_before = count_trainable(model)

model = inject_lora(model, rank=4, alpha=8.0)
trainable_after = count_trainable(model)

# LoRA should have far fewer trainable params than original
ratio = trainable_after / trainable_before
assert ratio < 0.5, f"LoRA should reduce trainable params significantly, ratio={ratio:.2f}"

print(f"Original trainable params: {trainable_before:,}")
print(f"LoRA trainable params: {trainable_after:,}")
print(f"Ratio: {ratio:.2%}")
print("Main Problem 2 passed.")

### Main Problem 3: Sliding Window Attention (30 min) — Review Problem

Implement `sliding_window_attention(q, k, v, window_size)` for efficient local attention.

**Interface:**
```python
def sliding_window_attention(
    q: torch.Tensor,  # (B, N, D)
    k: torch.Tensor,  # (B, N, D)
    v: torch.Tensor,  # (B, N, D)
    window_size: int,  # each position attends to [max(0, i-w), min(N, i+w+1))
) -> torch.Tensor:    # (B, N, D)
```

**Behavior:**
- Position `i` attends to positions in the range `[max(0, i - window_size), min(N, i + window_size + 1))`
- Standard scaled dot-product attention, but with a mask that limits each query to its local window
- Positions outside the window get `-inf` before softmax

**Implementation approach:**
1. Compute full attention scores: `scores = (q @ k.T) / sqrt(D)` → shape `(B, N, N)`
2. Create a window mask: for each position `i`, only positions `j` where `|i - j| <= window_size` are valid
3. Apply mask: set invalid positions to `-inf`
4. Softmax + matmul with V

**Notes:**
- This is the conceptually simple (but not memory-optimal) approach using masking. A real implementation would use block-sparse ops.
- `window_size=0` means each position only attends to itself
- `window_size >= N-1` means full attention (no masking)

In [ ]:
# Main Problem 3: Your implementation



In [ ]:
# Main Problem 3: Tests
torch.manual_seed(42)
B, N, D = 2, 16, 32

q = torch.randn(B, N, D)
k = torch.randn(B, N, D)
v = torch.randn(B, N, D)

# Test 1: Output shape
out = sliding_window_attention(q, k, v, window_size=3)
assert out.shape == (B, N, D), f"Expected {(B, N, D)}, got {out.shape}"

# Test 2: window_size >= N should give same result as full attention
out_full_window = sliding_window_attention(q, k, v, window_size=N)

# Standard full attention for comparison
scores_full = (q @ k.transpose(-2, -1)) / math.sqrt(D)
attn_full = F.softmax(scores_full, dim=-1)
out_full_ref = attn_full @ v

assert torch.allclose(out_full_window, out_full_ref, atol=1e-5), \
    "window_size=N should give same result as full attention"

# Test 3: window_size=0 should give self-attention only (each position attends only to itself)
out_self_only = sliding_window_attention(q, k, v, window_size=0)
# With window_size=0, each position only sees itself, so output should be v
# (softmax of a single element is 1.0, scaled_score @ v[i] = v[i])
assert torch.allclose(out_self_only, v, atol=1e-5), \
    "window_size=0 should give self-attention only (output = v)"

# Test 4: Smaller window should differ from full attention (for N > 1)
out_small_window = sliding_window_attention(q, k, v, window_size=1)
assert not torch.allclose(out_small_window, out_full_ref, atol=1e-3), \
    "Small window should differ from full attention"

# Test 5: Backward pass
q_grad = q.clone().requires_grad_(True)
out_grad = sliding_window_attention(q_grad, k, v, window_size=3)
out_grad.sum().backward()
assert q_grad.grad is not None, "Should support backward pass"

print("Main Problem 3 passed.")